In [6]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from PIL import Image

In [7]:
CSV_PATH = "../processing/master_with_paths.csv"
DATASET_ROOT = "../data"
ALLOWED_LIGHT_FOLDERS = {"Spot Light"}
MATERIAL = "PlasticGlossy"
BATCH = "Batch 1 - Cycles AGX"
NUM_ACTIVE_LIGHTS = 1

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
np.random.seed(42)
tf.random.set_seed(42)

In [8]:
df = pd.read_csv(CSV_PATH, low_memory=False)

filtered = df[
    df["light_folder"].isin(ALLOWED_LIGHT_FOLDERS)
    & (df["num_active_lights"].astype(int) == NUM_ACTIVE_LIGHTS)
    & (df["material_folder"].astype(str) == MATERIAL)
    & (df["batch_folder"].astype(str) == BATCH)
].copy()

cols = [
    "image_relpath",
    "shape_name",
    "material_folder",
    "light_folder",
    "batch_folder",
    "frame",
    "config_id",
    "camera_png",
    "camera_name",
    "cam_pos_x", "cam_pos_y", "cam_pos_z",
    "cam_forward_x", "cam_forward_y", "cam_forward_z",
    "cam_up_x", "cam_up_y", "cam_up_z",
    "cam_right_x", "cam_right_y", "cam_right_z",
    "focal_length_mm",
    "light0_energy", "light0_color_r", "light0_color_g", "light0_color_b",
    "light0_pos_x", "light0_pos_y", "light0_pos_z",
    "light0_dir_x", "light0_dir_y", "light0_dir_z",
    "light0_spot_cone_deg", "light0_spot_blend"
]

In [9]:
# convert to camera local coordinates with blender conventions

def normalize_rows(v: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.clip(n, eps, None)

def add_light_cam_coords_blender_local(df):
    cam_pos = df[["cam_pos_x", "cam_pos_y", "cam_pos_z"]].to_numpy(dtype=np.float32)

    cam_right = normalize_rows(df[["cam_right_x", "cam_right_y", "cam_right_z"]].to_numpy(dtype=np.float32))
    cam_up    = normalize_rows(df[["cam_up_x",    "cam_up_y",    "cam_up_z"]].to_numpy(dtype=np.float32))

    cam_forward = normalize_rows(df[["cam_forward_x", "cam_forward_y", "cam_forward_z"]].to_numpy(dtype=np.float32))
    cam_back = -cam_forward

    light_pos = df[["light0_pos_x", "light0_pos_y", "light0_pos_z"]].to_numpy(dtype=np.float32)
    light_dir = df[["light0_dir_x", "light0_dir_y", "light0_dir_z"]].to_numpy(dtype=np.float32)
    light_dir = normalize_rows(light_dir)

    rel = light_pos - cam_pos

    df["light0_pos_cam_x"] = np.einsum("ij,ij->i", rel, cam_right)
    df["light0_pos_cam_y"] = np.einsum("ij,ij->i", rel, cam_up)
    df["light0_pos_cam_z"] = np.einsum("ij,ij->i", rel, cam_back)

    df["light0_dir_cam_x"] = np.einsum("ij,ij->i", light_dir, cam_right)
    df["light0_dir_cam_y"] = np.einsum("ij,ij->i", light_dir, cam_up)
    df["light0_dir_cam_z"] = np.einsum("ij,ij->i", light_dir, cam_back)

    return df

cols_prepped = add_light_cam_coords_blender_local(filtered[cols].copy())

In [10]:
image_df = cols_prepped.copy()
image_df["image_path"] = image_df["image_relpath"].astype(str).apply(lambda p: os.path.join(DATASET_ROOT, p))

image_df = image_df[image_df["image_path"].map(os.path.exists)].reset_index(drop=True)
if image_df.empty:
    raise ValueError("No images found for current filters. Check DATASET_ROOT and image_relpath values.")

def load_and_preprocess_image(path: str) -> np.ndarray:
    with Image.open(path) as img:
        img = img.convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
        return np.asarray(img, dtype=np.float32) / 255.0

X_images = np.stack([load_and_preprocess_image(p) for p in image_df["image_path"]], axis=0)

In [ ]:
target_cols = [
    "light0_pos_cam_x", "light0_pos_cam_y", "light0_pos_cam_z",
    "light0_dir_cam_x", "light0_dir_cam_y", "light0_dir_cam_z",
]

# make sure no leakage from labeled features
exclude_cols = set(target_cols + [
    "image_relpath", "image_path", "camera_png",
    "light0_pos_x", "light0_pos_y", "light0_pos_z",
    "light0_dir_x", "light0_dir_y", "light0_dir_z",
])

known_df = image_df[[c for c in image_df.columns if c not in exclude_cols]].copy()
cat_cols = [c for c in ["shape_name", "material_folder", "light_folder", "batch_folder", "camera_name"] if c in known_df.columns]
known_df = pd.get_dummies(known_df, columns=cat_cols, drop_first=False)

X_known = known_df.to_numpy(dtype=np.float32)
y = image_df[target_cols].to_numpy(dtype=np.float32)

idx = np.arange(len(image_df))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)
idx_train, idx_val = train_test_split(idx_train, test_size=0.2, random_state=42)

X_img_train, X_img_val, X_img_test = X_images[idx_train], X_images[idx_val], X_images[idx_test]
X_tab_train, X_tab_val, X_tab_test = X_known[idx_train], X_known[idx_val], X_known[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]

tab_mean = X_tab_train.mean(axis=0, keepdims=True)
tab_std = X_tab_train.std(axis=0, keepdims=True)
tab_std[tab_std < 1e-8] = 1.0

X_tab_train = (X_tab_train - tab_mean) / tab_std
X_tab_val = (X_tab_val - tab_mean) / tab_std
X_tab_test = (X_tab_test - tab_mean) / tab_std

print("Train/Val/Test:", len(idx_train), len(idx_val), len(idx_test))
print("Image input shape:", X_img_train.shape[1:])
print("Tabular features:", X_tab_train.shape[1])

Train/Val/Test: 848 212 266
Image input shape: (224, 224, 3)
Tabular features: 31


In [12]:
img_input = keras.Input(shape=(*IMG_SIZE, 3), name="image")
x = layers.Conv2D(32, 3, activation="relu", padding="same")(img_input)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)

tab_input = keras.Input(shape=(X_tab_train.shape[1],), name="known_params")
t = layers.Dense(128, activation="relu")(tab_input)
t = layers.Dense(64, activation="relu")(t)

h = layers.Concatenate()([x, t])
h = layers.Dense(128, activation="relu")(h)
h = layers.Dropout(0.2)(h)
out = layers.Dense(6, name="light_cam_pose")(h)

model = keras.Model(inputs=[img_input, tab_input], outputs=out)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
model.summary()

callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

history = model.fit(
    {"image": X_img_train, "known_params": X_tab_train},
    y_train,
    validation_data=({"image": X_img_val, "known_params": X_tab_val}, y_val),
    epochs=30,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

test_loss, test_mae = model.evaluate({"image": X_img_test, "known_params": X_tab_test}, y_test, verbose=0)
y_pred = model.predict({"image": X_img_test, "known_params": X_tab_test}, verbose=0)

pos_mae = np.mean(np.abs(y_pred[:, :3] - y_test[:, :3]))
pred_dir = y_pred[:, 3:6]
true_dir = y_test[:, 3:6]
pred_dir = pred_dir / np.clip(np.linalg.norm(pred_dir, axis=1, keepdims=True), 1e-8, None)
true_dir = true_dir / np.clip(np.linalg.norm(true_dir, axis=1, keepdims=True), 1e-8, None)
cosang = np.sum(pred_dir * true_dir, axis=1)
cosang = np.clip(cosang, -1.0, 1.0)
angle_err_deg = np.degrees(np.arccos(cosang))

print(f"Test loss: {test_loss:.6f}")
print(f"Test MAE (all 6 outputs): {test_mae:.6f}")
print(f"Position MAE (camera xyz): {pos_mae:.6f}")
print(f"Direction angular error mean (deg): {angle_err_deg.mean():.3f}")
print(f"Direction angular error median (deg): {np.median(angle_err_deg):.3f}")

2026-02-15 15:04:39.510919: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_UNKNOWN: unknown error
2026-02-15 15:04:39.510953: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-02-15 15:04:39.510958: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: katelyns-pc
2026-02-15 15:04:39.510961: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] hostname: katelyns-pc
2026-02-15 15:04:39.511065: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:190] libcuda reported version is: 570.211.1
2026-02-15 15:04:39.511080: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:194] kernel reported version is: 570.211

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        896 │ image[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ known_params        │ (None, 31)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │      4,096 │ known_params[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     24,704 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ light_cam_pose      │ (None, 6)         │        774 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 147,590 (576.52 KB)

 Trainable params: 147,590 (576.52 KB)

...

Epoch 1/30


2026-02-15 15:04:40.081173: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 510590976 exceeds 10% of free system memory.
2026-02-15 15:04:41.558850: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 205520896 exceeds 10% of free system memory.
2026-02-15 15:04:42.147406: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 205520896 exceeds 10% of free system memory.


 1/27 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - loss: 2.4589 - mae: 1.1383

2026-02-15 15:04:42.380525: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 205520896 exceeds 10% of free system memory.
2026-02-15 15:04:42.833494: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 205520896 exceeds 10% of free system memory.


27/27 ━━━━━━━━━━━━━━━━━━━━ 20s 684ms/step - loss: 1.2488 - mae: 0.7753 - val_loss: 0.2772 - val_mae: 0.3807
Epoch 2/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 677ms/step - loss: 0.2487 - mae: 0.3691 - val_loss: 0.0879 - val_mae: 0.1981
Epoch 3/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 680ms/step - loss: 0.1331 - mae: 0.2723 - val_loss: 0.0283 - val_mae: 0.1152
Epoch 4/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 19s 689ms/step - loss: 0.0898 - mae: 0.2296 - val_loss: 0.0125 - val_mae: 0.0848
Epoch 5/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 19s 691ms/step - loss: 0.0745 - mae: 0.2088 - val_loss: 0.0104 - val_mae: 0.0793
Epoch 6/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 670ms/step - loss: 0.0682 - mae: 0.1986 - val_loss: 0.0111 - val_mae: 0.0828
Epoch 7/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 668ms/step - loss: 0.0626 - mae: 0.1876 - val_loss: 0.0072 - val_mae: 0.0651
Epoch 8/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 672ms/step - loss: 0.0564 - mae: 0.1781 - val_loss: 0.0069 - val_mae: 0.0618
Epoch 9/30
27/27 ━━━━━━━━━━━━━━━━━━━━ 18s 673ms/step - loss

In [14]:
# save weights for singular testing
model.save("angular_predictor.keras")

In [17]:
# test on singular unseen image .. Cylinder/PlasticGlossy/Spot Light/Batch 3 - Cycles Filmic/1.png
test_image_path = os.path.join(DATASET_ROOT, "Cylinder", "PlasticGlossy", "Spot Light", "Batch 3 - Cycles Filmic", "1.png")
if not os.path.exists(test_image_path):
    raise FileNotFoundError(f"Image not found: {test_image_path}")

# build image input
test_image = load_and_preprocess_image(test_image_path)
X_img_single = np.expand_dims(test_image, axis=0)

# match csv row, then convert to camera coordinates with earlier function
test_relpath = os.path.relpath(test_image_path, DATASET_ROOT).replace("\\", "/")
row_match = df[df["image_relpath"].astype(str).str.replace("\\", "/", regex=False) == test_relpath].copy()
if row_match.empty:
    raise ValueError(f"No CSV row found for image_relpath={test_relpath}")
if len(row_match) > 1:
    print(f"Found {len(row_match)} rows for this image_relpath; using the first match.")
row_match = row_match.iloc[[0]].copy()

row_cam = add_light_cam_coords_blender_local(row_match[cols].copy())
y_true_single = row_cam[target_cols].iloc[0].to_numpy(dtype=np.float32)

# build known-parameter input exactly like training
known_single = row_cam[[c for c in row_cam.columns if c not in exclude_cols]].copy()
single_cat_cols = [c for c in cat_cols if c in known_single.columns]
known_single = pd.get_dummies(known_single, columns=single_cat_cols, drop_first=False)
known_single = known_single.reindex(columns=known_df.columns, fill_value=0.0)

X_tab_single = known_single.to_numpy(dtype=np.float32)
X_tab_single = (X_tab_single - tab_mean) / tab_std
if X_tab_single.shape[0] != X_img_single.shape[0]:
    raise ValueError(f"Batch mismatch: image batch={X_img_single.shape[0]}, tabular batch={X_tab_single.shape[0]}")

# predict
y_pred_single = model.predict([X_img_single, X_tab_single], verbose=0)[0]

# compare prediction vs converted camera-space truth
pred_pos = y_pred_single[:3]
true_pos = y_true_single[:3]
pred_dir = y_pred_single[3:6]
true_dir = y_true_single[3:6]

pred_dir = pred_dir / np.clip(np.linalg.norm(pred_dir), 1e-8, None)
true_dir = true_dir / np.clip(np.linalg.norm(true_dir), 1e-8, None)
cosang = np.clip(np.dot(pred_dir, true_dir), -1.0, 1.0)
angle_err_deg = float(np.degrees(np.arccos(cosang)))
pos_abs_err = np.abs(pred_pos - true_pos)

print("Image:", test_relpath)
print("Predicted pos (cam xyz):", pred_pos)
print("True pos (cam xyz):     ", true_pos)
print("Abs pos error (xyz):    ", pos_abs_err)
print("Predicted dir (cam xyz):", pred_dir)
print("True dir (cam xyz):     ", true_dir)
print(f"Direction angle error (deg): {angle_err_deg:.3f}")

Found 13 rows for this image_relpath; using the first match.
Image: Cylinder/PlasticGlossy/Spot Light/Batch 3 - Cycles Filmic/1.png
Predicted pos (cam xyz): [-1.6258845  3.2231236 -2.0814404]
True pos (cam xyz):      [-1.5572329  2.747852  -1.6964074]
Abs pos error (xyz):     [0.06865168 0.47527146 0.385033  ]
Predicted dir (cam xyz): [ 0.39842638 -0.8099183  -0.43045187]
True dir (cam xyz):      [ 0.43966153 -0.7847284  -0.43692002]
Direction angle error (deg): 2.794
